# 🏗️ Pipeline de Preprocesamiento (Sprint 2)

## 🎯 Objetivo

Construir un pipeline reproducible que integre:

- Data Cleaning (Notebook 05)
- Feature Engineering (Notebook 06)
- Preparación para modelado (Notebook 07)

## ⚙️ Tecnologías

- scikit-learn Pipeline
- ColumnTransformer
- joblib

---

## 🚨 Regla clave

El pipeline debe:

✔ Entrenarse con `.fit()`  
✔ Aplicarse con `.transform()`  
✔ Evitar data leakage  
✔ Ser reutilizable en modelado (Sprint 3 y 4)

In [3]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

import joblib

## 📂 Carga de datos crudos

In [4]:
# ============================================================
# CARGAR DATA RAW
# ============================================================

df = pd.read_csv('../data/raw/01-hotel_bookings.csv')

print("Shape original:", df.shape)
df.head()

Shape original: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,03-07-15


## 🧹 Paso 1: Data Cleaning

Aplicamos las reglas definidas en el Notebook 05:
- Eliminación de leakage
- Eliminación de duplicados
- Corrección de nulos
- Eliminación de registros inválidos

In [5]:
# ============================================================
# DATA CLEANING
# ============================================================

df_clean = df.copy()

# eliminar leakage
df_clean.drop(['reservation_status', 'reservation_status_date', 'company'], axis=1, inplace=True)

# duplicados
df_clean = df_clean.drop_duplicates()

# sin personas
df_clean = df_clean[~(
    (df_clean['adults'] == 0) &
    (df_clean['children'].fillna(0) == 0) &
    (df_clean['babies'] == 0)
)]

# sin noches
df_clean = df_clean[~(
    (df_clean['stays_in_week_nights'] == 0) &
    (df_clean['stays_in_weekend_nights'] == 0)
)]

# nulos
df_clean['children'] = df_clean['children'].fillna(0)
df_clean['country'] = df_clean['country'].fillna(df_clean['country'].mode()[0])
df_clean['agent'] = df_clean['agent'].fillna(0)

# adr
df_clean = df_clean[df_clean['adr'] >= 0]
df_clean['adr'] = df_clean['adr'].clip(upper=df_clean['adr'].quantile(0.99))

print("Shape después de cleaning:", df_clean.shape)

Shape después de cleaning: (86372, 29)


## 🧠 Paso 2: Feature Engineering

Se generan variables derivadas para mejorar la capacidad predictiva del modelo.

In [6]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

df_feat = df_clean.copy()

df_feat['total_nights'] = df_feat['stays_in_week_nights'] + df_feat['stays_in_weekend_nights']
df_feat['total_guests'] = df_feat['adults'] + df_feat['children'] + df_feat['babies']
df_feat['total_price'] = df_feat['adr'] * df_feat['total_nights']
df_feat['price_per_person'] = df_feat['adr'] / df_feat['total_guests'].replace(0, np.nan)

df_feat['is_loyal'] = (df_feat['is_repeated_guest'] == 1).astype(int)

df_feat['adr_log'] = np.log1p(df_feat['adr'])
df_feat['lead_time_log'] = np.log1p(df_feat['lead_time'])

print("Shape después de FE:", df_feat.shape)

Shape después de FE: (86372, 36)


## 🎯 Separación de variables

In [7]:
# ============================================================
# X / y
# ============================================================

X = df_feat.drop('is_canceled', axis=1)
y = df_feat['is_canceled']

## ⚙️ Paso 3: Construcción del Pipeline

Se construye un pipeline usando:

- ColumnTransformer
- Imputación
- Escalado
- Encoding

In [8]:
# ============================================================
# DEFINIR COLUMNAS
# ============================================================

num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include='object').columns

C:\Users\Renzo\AppData\Local\Temp\ipykernel_5132\1290103767.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include='object').columns


In [9]:
# ============================================================
# PIPELINE NUMÉRICO
# ============================================================

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [10]:
# ============================================================
# PIPELINE CATEGÓRICO
# ============================================================

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [11]:
# ============================================================
# COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

## 🚀 Entrenamiento del pipeline

In [12]:
# ============================================================
# FIT PIPELINE
# ============================================================

preprocessor.fit(X)

print("Pipeline entrenado correctamente")

Pipeline entrenado correctamente


## 🔄 Transformación de datos

In [13]:
# ============================================================
# TRANSFORM
# ============================================================

X_transformed = preprocessor.transform(X)

print("Transformación aplicada")
print("Shape:", X_transformed.shape)

Transformación aplicada
Shape: (86372, 261)


## 💾 Guardado del pipeline

In [16]:
# ============================================================
# GUARDAR PIPELINE (VERSIÓN ROBUSTA)
# ============================================================

import os
import joblib

# Crear carpeta si no existe
os.makedirs('../models', exist_ok=True)

# Guardar pipeline
joblib.dump(preprocessor, '../models/preprocessor.pkl')

print("✅ Pipeline guardado en models/")

✅ Pipeline guardado en models/


## ✅ Conclusión

Se construyó un pipeline completo que:

✔ Integra cleaning + feature engineering  
✔ Es reproducible  
✔ Evita data leakage  
✔ Está listo para ser usado en modelos (Sprint 3)

👉 Este pipeline será utilizado en:

- 09_baseline_models.ipynb
- 10_evaluation.ipynb